In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import time
import sys
from typing import Tuple, Optional, Sequence
from modules import utils

# from Code.PTB_data_utils import PTB_word, load_PTB_word

In [ ]:
class LSTM(nn.Module):
    def __init__(self,
                 incoming,
                 num_units,
                 ingate=None,
                 forgetgate=None,
                 cell=None,
                 outgate=None,
                 hid_init=0.0,
                 cell_init=0.0,
                 learn_init=True,
                 nonlinearity=torch.tanh,
                 backwards=False,
                 gradient_steps=-1,
                 mask_input=None,
                 only_return_final=False,
                 hid_prop=False,
                 **kwargs):
        
        incomings = incoming if hid_prop else [incoming]
        self.mask_incoming_index = -1
        if mask_input is not None:
            incomings.append(mask_input)
            self.mask_incoming_index = len(incomings)-1
        super().__init__()

        self.name = "LSTM"
        self.nonlinearity = nonlinearity
        self.num_units = num_units
        
        if isinstance(incoming, int):
            self.num_inputs = incoming
        elif isinstance(incoming, (tuple, list)):
            self.num_inputs = int(np.prod(incoming[2:])) if len(incoming) > 2 else incoming[-1]
        else:
            raise ValueError("incoming must be int or shape-like (tuple/list)")
        
        self.backwards = backwards
        self.gradient_steps = gradient_steps
        self.only_return_final = only_return_final
        self._srng = torch.Generator().manual_seed(np.random.randint(1e7))
        self.hidden_noise = torch.ones(1, dtype=torch.float32)
        self.hidden_clip = torch.ones(1, dtype=torch.float32)
        self.mu_hid = torch.ones(1, dtype=torch.float32)
        self.log_sigma2_hid = torch.ones(1, dtype=torch.float32)
        self.learn_init = learn_init
        self.hid_prop = hid_prop

        if ingate is None:
            self.W_in_to_ingate = nn.Parameter(torch.empty(self.num_inputs, self.num_units))
            self.W_hid_to_ingate = nn.Parameter(torch.empty(self.num_units, self.num_units))
            self.b_ingate = nn.Parameter(torch.full((self.num_units,), 0.0, dtype=torch.float32)) # Constant(0.)
            nn.init.xavier_uniform_(self.W_in_to_ingate) # GlorotUniform
            nn.init.orthogonal_(self.W_hid_to_ingate, gain=1.1) # Orthogonal
            self.nonlinearity_ingate = utils.hard_sigmoid
        
        if forgetgate is None:
            self.W_in_to_forgetgate = nn.Parameter(torch.empty(self.num_inputs, self.num_units))
            self.W_hid_to_forgetgate = nn.Parameter(torch.empty(self.num_units, self.num_units))
            self.b_forgetgate = nn.Parameter(torch.full((self.num_units,), 1.0, dtype=torch.float32))  # Constant(1.)
            nn.init.xavier_uniform_(self.W_in_to_forgetgate)
            nn.init.orthogonal_(self.W_hid_to_forgetgate, gain=1.1)
            self.nonlinearity_forgetgate = utils.hard_sigmoid

        if cell is None:
            self.W_in_to_cell = nn.Parameter(torch.empty(self.num_inputs, self.num_units))
            self.W_hid_to_cell = nn.Parameter(torch.empty(self.num_units, self.num_units))
            self.b_cell = nn.Parameter(torch.full((self.num_units,), 0.0, dtype=torch.float32))
            nn.init.xavier_uniform_(self.W_in_to_cell)
            nn.init.orthogonal_(self.W_hid_to_cell, gain=1.1)
            self.nonlinearity_cell = torch.tanh

        if outgate is None:
            self.W_in_to_outgate = nn.Parameter(torch.empty(self.num_inputs, self.num_units))
            self.W_hid_to_outgate = nn.Parameter(torch.empty(self.num_units, self.num_units))
            self.b_outgate = nn.Parameter(torch.full((self.num_units,), 0.0, dtype=torch.float32))
            nn.init.xavier_uniform_(self.W_in_to_outgate)
            nn.init.orthogonal_(self.W_hid_to_outgate, gain=1.1)
            self.nonlinearity_outgate = utils.hard_sigmoid

        self.hid_init = nn.Parameter(
            torch.full((1, self.num_units), hid_init, dtype=torch.float32),
            requires_grad=learn_init
        )

        self.cell_init = nn.Parameter(
            torch.full((1, self.num_units), cell_init, dtype=torch.float32),
            requires_grad=learn_init
        )
    
    def input_preactivation(self, input: torch.Tensor, gate_type: str) -> torch.Tensor:
        if gate_type == 'input':
            return input @ self.W_in_to_ingate
        elif gate_type == 'forget':
            return input @ self.W_in_to_forgetgate
        elif gate_type == 'cell':
            return input @ self.W_in_to_cell
        elif gate_type == 'output':
            return input @ self.W_in_to_outgate
        else:
            raise ValueError(f"Unknown gate_type: {gate_type}")
    
    def get_output_shape_for(self, input_shape):
        """
        input_shape: tuple, (batch_size, seq_len, input_size)
        """
        batch_size, seq_len = input_shape[0], input_shape[1]

        if self.only_return_final:
            return (batch_size, self.num_units)
        elif self.hid_prop:
            return (2, batch_size, seq_len, self.num_units)
        else:
            return (batch_size, seq_len, self.num_units)
    
    def generate_noise_and_clip(self, num_batch, seq_len, **kwargs):
        return
    
    def forward(self, inputs, deterministic: bool = False, clip: bool = False, **kwargs):
        """
        PyTorch forward, сохраняет логику оригинального get_output_for.
        inputs: либо тензор (batch, seq_len, input_dim) либо список/tuple, где inputs[0] - вход,
                и при self.mask_incoming_index > 0 mask находится в inputs[self.mask_incoming_index].
        Возвращает:
        - если self.only_return_final: (batch, num_units)
        - elif self.hid_prop: (2, batch, seq_len, num_units)
        - else: (batch, seq_len, num_units)
        """
        if isinstance(inputs, (list, tuple)):
            input = inputs[0]
        else:
            input = inputs

        mask = None
        if getattr(self, "mask_incoming_index", -1) > 0:
            if isinstance(inputs, (list, tuple)) and len(inputs) > self.mask_incoming_index:
                mask = inputs[self.mask_incoming_index]
            else:
                mask = None

        input = input.transpose(0, 1)
        seq_len, num_batch, _ = input.shape

        # Генерация шума/клипов, если нужно (метод должен быть реализован)
        # передаём num_batch и seq_len, а также флаги deterministic/clip для совместимости
        try:
            self.generate_noise_and_clip(num_batch, deterministic, clip)
        except TypeError:
            # если generate_noise_and_clip не ожидает clip/deterministic — вызовем только по num_batch
            self.generate_noise_and_clip(num_batch, seq_len)

        # Предактивации входной части (возвращают (seq_len, batch, num_units))
        input_i = self.input_preactivation(input, 'input', deterministic=deterministic, clip=clip, **kwargs) + self.b_ingate
        input_f = self.input_preactivation(input, 'forget', deterministic=deterministic, clip=clip, **kwargs) + self.b_forgetgate
        input_c = self.input_preactivation(input, 'cell', deterministic=deterministic, clip=clip, **kwargs) + self.b_cell
        input_o = self.input_preactivation(input, 'output', deterministic=deterministic, clip=clip, **kwargs) + self.b_outgate

        # Начальные состояния
        if self.hid_prop:
            # ожидаем, что inputs[1] содержит (2, batch, num_units) или (2, num_batch, num_units)
            hid_init, cell_init = inputs[1][0], inputs[1][1]
        else:
            hid_init = self.hid_init.expand(num_batch, -1).to(input.device, dtype=input.dtype)   # (batch, num_units)
            cell_init = self.cell_init.expand(num_batch, -1).to(input.device, dtype=input.dtype)

        hid = hid_init
        cell = cell_init

        hid_seq = []
        cell_seq = []

        # если есть маска, приведение её к форме (seq_len, batch, 1) для индексации
        if mask is not None:
            # mask может быть (batch, seq_len) или (batch, seq_len, 1)
            if mask.ndim == 3:
                mask_seq = mask.permute(1, 0, 2)  # (seq_len, batch, 1)
            else:
                mask_seq = mask.unsqueeze(-1).permute(1, 0, 2)  # (seq_len, batch, 1)
            mask_seq = mask_seq.to(dtype=input.dtype, device=input.device)
        else:
            mask_seq = None

        for t in range(seq_len):
            input_n_i = input_i[t]
            input_n_f = input_f[t]
            input_n_c = input_c[t]
            input_n_o = input_o[t]

            # рекуррентные предактивации от предыдущего hid (должен быть реализован)
            hid_preact_i = self.hidden_preactivation(hid, 'input', **kwargs)
            hid_preact_f = self.hidden_preactivation(hid, 'forget', **kwargs)
            hid_preact_c = self.hidden_preactivation(hid, 'cell', **kwargs)
            hid_preact_o = self.hidden_preactivation(hid, 'output', **kwargs)

            ingate = self.nonlinearity_ingate(input_n_i + hid_preact_i)
            forgetgate = self.nonlinearity_forgetgate(input_n_f + hid_preact_f)
            cell_candidate = self.nonlinearity_cell(input_n_c + hid_preact_c)
            cell = forgetgate * cell + ingate * cell_candidate
            outgate = self.nonlinearity_outgate(input_n_o + hid_preact_o)
            hid = outgate * self.nonlinearity(cell)

            # маскирование (T.switch(mask_n, new, prev) эквивалентно m*new + (1-m)*prev)
            if mask_seq is not None:
                m = mask_seq[t]  # (batch, 1)
                # broadcasting: m * new + (1-m) * previous
                cell = m * cell + (1.0 - m) * cell_init
                hid  = m * hid  + (1.0 - m) * hid_init

            hid_seq.append(hid)
            cell_seq.append(cell)

        # собираем последовательности вне цикла
        cell_out = torch.stack(cell_seq, dim=0)  # (seq_len, batch, num_units)
        hid_out = torch.stack(hid_seq, dim=0)    # (seq_len, batch, num_units)

        # Если only_return_final -> финальное скрытое состояние (последний по времени, в порядке вычисления)
        if self.only_return_final:
            return hid_out[-1]   # (batch, num_units)

        # если вычисляли backwards, нужно вернуть в прямой порядок времени (как в Theano)
        if self.backwards:
            hid_out = hid_out.flip(dims=[0])
            cell_out = cell_out.flip(dims=[0])

        # переставим в (batch, seq_len, num_units)
        hid_out = hid_out.transpose(0, 1)   # (batch, seq_len, num_units)
        cell_out = cell_out.transpose(0, 1)

        if self.hid_prop:
            # вернуть в виде (2, batch, seq_len, num_units)
            return torch.cat([hid_out.unsqueeze(0), cell_out.unsqueeze(0)], dim=0)
        else:
            return hid_out


class BayesianLSTM(LSTM):
    """
    config: L probabilistic weight with lognormal prior, N probabilistic weight with standart normal prior, 
           D deterministic learnable weight, 
           C constant weight (1 for multiplicative and 0 for additive weights)\
           
           config[0]: W input_to_hidden, W hidden_to_hidden
                     L N D (C is not supported)
           config[1]: hat Z preactivation multiplicative weights
                     L N D C
           config[2]: Z input and hidden multiplicative weights
                     L N D C I R
    """
    def __init__(self, 
                 incoming, 
                 num_units,
                 log_sigma_in_init = -3.0, 
                 log_sigma_hid_init = -3.0,
                 ingate=None,
                 forgetgate=None,
                 cell=None,
                 outgate=None,
                 hid_init=0.0,
                 cell_init=0.0,
                 learn_init=True,
                 nonlinearity=torch.tanh,
                 backwards=False,
                 gradient_steps=-1,
                 mask_input=None,
                 only_return_final=False,
                 hid_prop = False,
                 config="DCC"):
 
        super().__init__(incoming, 
                         num_units, 
                         ingate, 
                         forgetgate, 
                         cell, 
                         outgate,
                         hid_init, 
                         cell_init, 
                         learn_init, 
                         nonlinearity, 
                         backwards, 
                         gradient_steps, 
                         mask_input,
                         only_return_final,
                         hid_prop)
        
        self.name = 'BayesianLSTM'
        self.reg = True
        self.config = config
        self.log_sigma_in_init = log_sigma_in_init
        self.log_sigma_hid_init = log_sigma_hid_init
        self.dtype = torch.float32

        if self.config[0] in {"L", "N"}:
            self.logsig_w_in = nn.Parameter(torch.full((4, incoming, num_units), log_sigma_in_init))
            self.logsig_w_hid = nn.Parameter(torch.full((4, num_units, num_units), log_sigma_hid_init))
        else:
            self.register_buffer("logsig_w_in", torch.zeros(4))
            self.register_buffer("logsig_w_hid", torch.zeros(4))
        
        if self.config[2] in {"L", "N", "D", "I"}:
            self.mu_in = nn.Parameter(torch.ones(incoming))
        if self.config[2] in {"L", "N", "I"}:
            self.logsig_in = nn.Parameter(torch.full((incoming,), log_sigma_in_init))
        if self.config[2] in {"L", "N", "D", "R"}:
            self.mu_hid = nn.Parameter(torch.ones(num_units))
        if self.config[2] in {"L", "N", "R"}:
            self.logsig_hid = nn.Parameter(torch.full((num_units,), log_sigma_hid_init))

        if self.config[1] in {"L", "N", "D"}:
            self.mu_gates = nn.Parameter(torch.ones(4, num_units))
        if self.config[1] in {"L", "N"}:
            self.logsig_gates = nn.Parameter(torch.full((4, num_units), log_sigma_hid_init))

        self.input_noise = None
        self.hidden_noise = None
        self.input_clip = None
        self.hidden_clip = None

        self.thresh = 3.0
    
    def generate_noise_and_clip(self, num_batch, deterministic=False, clip=False):
        with torch.no_grad():
            if not deterministic:
                # --- W_in, W_hid noise ---
                if self.config[0] in {"L", "N"}:
                    self.input_w_noise = torch.randn(4, self.num_inputs, self.num_units) * torch.exp(self.logsig_w_in)
                    self.hidden_w_noise = torch.randn(4, self.num_units, self.num_units) * torch.exp(self.logsig_w_hid)
                else:
                    self.input_w_noise = torch.zeros(4)
                    self.hidden_w_noise = torch.zeros(4)

                # --- input/hidden noise ---
                if self.config[2] in {"L", "N"}:
                    self.input_noise = torch.randn(num_batch, self.num_inputs) * torch.exp(self.logsig_in) + self.mu_in
                    self.hidden_noise = torch.randn(num_batch, self.num_units) * torch.exp(self.logsig_hid) + self.mu_hid
                elif self.config[2] == "I":
                    self.input_noise = torch.randn(num_batch, self.num_inputs) * torch.exp(self.logsig_in) + self.mu_in
                    self.hidden_noise = torch.ones(1)
                elif self.config[2] == "R":
                    self.input_noise = torch.ones(1)
                    self.hidden_noise = torch.randn(num_batch, self.num_units) * torch.exp(self.logsig_hid) + self.mu_hid
                elif self.config[2] == "D":
                    self.input_noise = self.mu_in
                    self.hidden_noise = self.mu_hid
                else:
                    self.input_noise = torch.ones(1)
                    self.hidden_noise = torch.ones(1)
            
                # --- gates noise ---
                if self.config[1] in {"L", "N"}:
                    self.gates_noise = torch.randn(4, num_batch, self.num_units) * torch.exp(self.logsig_gates)[:, None, :] + self.mu_gates[:, None, :]
                elif self.config[1] == "D":
                    self.gates_noise = self.mu_gates
                else:
                    self.gates_noise = torch.ones(4)
            
            else:
                # deterministic path
                self.input_w_noise = torch.zeros(4)
                self.hidden_w_noise = torch.zeros(4)

                if self.config[2] in {"L", "N", "D"}:
                    self.input_noise = self.mu_in
                    self.hidden_noise = self.mu_hid
                elif self.config[2] == "I":
                    self.input_noise = self.mu_in
                    self.hidden_noise = torch.ones(1)
                elif self.config[2] == "R":
                    self.input_noise = torch.ones(1)
                    self.hidden_noise = self.mu_hid
                else:
                    self.input_noise = torch.ones(1)
                    self.hidden_noise = torch.ones(1)

                if self.config[1] in {"L", "N", "D"}:
                    self.gates_noise = self.mu_gates
                else:
                    self.gates_noise = torch.ones(4, dtype=self.dtype)
            
            if clip:
                if self.config[0] == "L":
                    W_in_cat = torch.cat([self.W_in_to_ingate[None,:,:],
                                        self.W_in_to_forgetgate[None,:,:],
                                        self.W_in_to_cell[None,:,:],
                                        self.W_in_to_outgate[None,:,:]], dim=0)
                    log_alpha_w_in = utils.clip_func(2*self.logsig_w_in - torch.log(W_in_cat**2 + self.epsilon))
                    self.input_w_clip = log_alpha_w_in <= self.thresh

                    W_hid_cat = torch.cat([self.W_hid_to_ingate[None,:,:],
                                        self.W_hid_to_forgetgate[None,:,:],
                                        self.W_hid_to_cell[None,:,:],
                                        self.W_hid_to_outgate[None,:,:]], dim=0)
                    log_alpha_w_hid = utils.clip_func(2*self.logsig_w_hid - utils.safe_torch_log(W_hid_cat**2))
                    self.hidden_w_clip = log_alpha_w_hid <= self.thresh
                else:
                    self.input_w_clip = torch.ones(4)
                    self.hidden_w_clip = torch.ones(4)

                if self.config[2] == "L":
                    log_alpha_in = utils.clip_func(2*self.logsig_in - utils.safe_torch_log(self.mu_in**2))
                    self.input_clip = log_alpha_in <= self.thresh
                    log_alpha_hid = utils.clip_func(2*self.logsig_hid - utils.safe_torch_log(self.mu_hid**2))
                    self.hidden_clip = log_alpha_hid <= self.thresh
                elif self.config[2] == "I":
                    log_alpha_in = utils.clip_func(2*self.logsig_in - utils.safe_torch_log(self.mu_in**2))
                    self.input_clip = log_alpha_in <= self.thresh
                    self.hidden_clip = torch.ones(1)
                elif self.config[2] == "R":
                    self.input_clip = torch.ones(1)
                    log_alpha_hid = utils.clip_func(2*self.logsig_hid - utils.safe_torch_log(self.mu_hid**2))
                    self.hidden_clip = log_alpha_hid <= self.thresh
                else:
                    self.input_clip = torch.ones(1)
                    self.hidden_clip = torch.ones(1)

                if self.config[1] == "L":
                    log_alpha_gates = utils.clip_func(2*self.logsig_gates - utils.safe_torch_log(self.mu_gates**2))
                    self.gates_clip = log_alpha_gates <= self.thresh
                else:
                    self.gates_clip = torch.ones(4)
            
            else:
                self.input_w_clip = torch.ones(4)
                self.hidden_w_clip = torch.ones(4)
                self.input_clip = torch.ones(1)
                self.hidden_clip = torch.ones(1)
                self.gates_clip = torch.ones(4)

        return
    
    def input_preactivation(self, input, gate_type, deterministic = False, clip = False):

        if gate_type == "input":
            W, idx = self.W_in_to_ingate, 0
        elif gate_type == "forget":
            W, idx = self.W_in_to_forgetgate, 1
        elif gate_type == "cell":
            W, idx = self.W_in_to_cell, 2
        elif gate_type == "output":
            W, idx = self.W_in_to_outgate, 3
        else:
            raise ValueError(f"Unknown gate_type {gate_type}")
        
        input = input * self.input_noise * self.input_clip
        W_eff = (W + self.input_w_noise[idx]) * self.input_w_clip[idx]
        return torch.matmul(input, W_eff)
    
    def hidden_preactivation(self, hidden, gate_type, deterministic = False, clip = False):
        if gate_type == "input":
            W, idx = self.W_hid_to_ingate, 0
        elif gate_type == "forget":
            W, idx = self.W_hid_to_forgetgate, 1
        elif gate_type == "cell":
            W, idx = self.W_hid_to_cell, 2
        elif gate_type == "output":
            W, idx = self.W_hid_to_outgate, 3
        else:
            raise ValueError(f"Unknown gate_type {gate_type}")
        
        W_eff = (W + self.hidden_w_noise[idx]) * self.hidden_w_clip[idx]
        return torch.matmul(hidden, W_eff)
    
    def eval_reg(self, train_size):
        W_in = torch.cat([
            self.W_in_to_ingate.unsqueeze(0),
            self.W_in_to_forgetgate.unsqueeze(0),
            self.W_in_to_cell.unsqueeze(0),
            self.W_in_to_outgate.unsqueeze(0)
        ], dim=0)

        if self.config[0] == "N":
            KL_element_in = -self.logsig_w_in + 0.5 * (torch.exp(2 * self.logsig_w_in) + W_in**2) - 0.5
            KL = KL_element_in.sum()
        elif self.config[0] == "L":
            log_alpha_w_in = utils.clip_func(2 * self.logsig_w_in - utils.safe_torch_log(W_in**2))
            KL = utils.alpha_regf(log_alpha_w_in).sum()
        else:
            KL = torch.zeros(1, dtype=self.dtype, device=self.logsig_w_in.device).sum()

        # Скрытые веса
        W_hid = torch.cat([
            self.W_hid_to_ingate.unsqueeze(0),
            self.W_hid_to_forgetgate.unsqueeze(0),
            self.W_hid_to_cell.unsqueeze(0),
            self.W_hid_to_outgate.unsqueeze(0)
        ], dim=0)

        if self.config[0] == "N":
            KL_element_hid = -self.logsig_w_hid + 0.5 * (torch.exp(2 * self.logsig_w_hid) + W_hid**2) - 0.5
            KL += KL_element_hid.sum()
        elif self.config[0] == "L":
            log_alpha_w_hid = utils.clip_func(2 * self.logsig_w_hid - utils.safe_torch_log(W_hid**2))
            KL += utils.alpha_regf(log_alpha_w_hid).sum()

        # Нейроны
        if self.config[2] == "L":
            log_alpha_hid = utils.clip_func(2 * self.logsig_hid - utils.safe_torch_log(self.mu_hid**2))
            KL += utils.alpha_regf(log_alpha_hid).sum()
            log_alpha_in = utils.clip_func(2 * self.logsig_in - utils.safe_torch_log(self.mu_in**2))
            KL += utils.alpha_regf(log_alpha_in).sum()
        elif self.config[2] == "I":
            log_alpha_in = utils.clip_func(2 * self.logsig_in - utils.safe_torch_log(self.mu_in**2))
            KL += utils.alpha_regf(log_alpha_in).sum()
        elif self.config[2] == "R":
            log_alpha_hid = utils.clip_func(2 * self.logsig_hid - utils.safe_torch_log(self.mu_hid**2))
            KL += utils.alpha_regf(log_alpha_hid).sum()
        elif self.config[2] == "N":
            KL_element = -self.logsig_hid + 0.5 * (torch.exp(2 * self.logsig_hid) + self.mu_hid**2) - 0.5
            KL += KL_element.sum()
            KL_element = -self.logsig_in + 0.5 * (torch.exp(2 * self.logsig_in) + self.mu_in**2) - 0.5
            KL += KL_element.sum()

        # Гейты
        if self.config[1] == "L":
            log_alpha_gates = utils.clip_func(2 * self.logsig_gates - utils.safe_torch_log(self.mu_gates**2))
            KL += utils.alpha_regf(log_alpha_gates).sum()
        elif self.config[1] == "N":
            KL_element = -self.logsig_gates + 0.5 * (torch.exp(2 * self.logsig_gates) + self.mu_gates**2) - 0.5
            KL += KL_element.sum()
        reg = KL / train_size
        return reg
    
    def get_ard(self):
        # --- W ---
        if self.config[0] == "L":
            W_in = torch.cat([
                self.W_in_to_ingate.unsqueeze(0),
                self.W_in_to_forgetgate.unsqueeze(0),
                self.W_in_to_cell.unsqueeze(0),
                self.W_in_to_outgate.unsqueeze(0)
            ], dim=0)
            log_alpha_w_in = 2 * self.logsig_w_in - 2 * utils.safe_torch_log(torch.abs(W_in))
            mask_w_in = log_alpha_w_in < self.thresh

            W_hid = torch.cat([
                self.W_hid_to_ingate.unsqueeze(0),
                self.W_hid_to_forgetgate.unsqueeze(0),
                self.W_hid_to_cell.unsqueeze(0),
                self.W_hid_to_outgate.unsqueeze(0)
            ], dim=0)
            log_alpha_w_hid = 2 * self.logsig_w_hid - 2 * utils.safe_torch_log(torch.abs(W_hid))
            mask_w_hid = log_alpha_w_hid < self.thresh
        else:
            mask_w_in = torch.ones((4,) + self.W_in_to_ingate.shape, dtype=torch.bool, device=self.W_in_to_ingate.device)
            mask_w_hid = torch.ones((4,) + self.W_hid_to_ingate.shape, dtype=torch.bool, device=self.W_hid_to_ingate.device)

        # --- neurons ---
        mask_in = mask_w_in.any(dim=2).any(dim=0)
        mask_hid_by_w = mask_w_hid.any(dim=2).any(dim=0)
        mask_hid_by_z = torch.ones_like(mask_hid_by_w, dtype=torch.bool)
        
        def log_alpha_calc(logsig, mu):
            return 2 * logsig - 2 * utils.safe_torch_log(torch.abs(mu))

        if self.config[2] == "L":
            log_alpha_hid = log_alpha_calc(self.logsig_hid, self.mu_hid)
            log_alpha_in = log_alpha_calc(self.logsig_in, self.mu_in)
            mask_in = torch.logical_and(log_alpha_in < self.thresh, mask_in)
            mask_hid_by_z = log_alpha_hid < self.thresh
        elif self.config[2] == "I":
            log_alpha_in = log_alpha_calc(self.logsig_in, self.mu_in)
            mask_in = torch.logical_and(log_alpha_in < self.thresh, mask_in)
        elif self.config[2] == "R":
            log_alpha_hid = log_alpha_calc(self.logsig_hid, self.mu_hid)
            mask_hid_by_z = log_alpha_hid < self.thresh

        # --- gates ---
        mask = torch.cat([mask_w_in, mask_w_hid], dim=1)
        if self.config[1] == "L":
            log_alpha_gates = log_alpha_calc(self.logsig_gates, self.mu_gates)
            mask_gates = torch.logical_and(log_alpha_gates < self.thresh, mask.any(dim=1))
        else:
            mask_gates = mask.any(dim=1)

        return {
            "w_input": mask_w_in,
            "w_hidden": mask_w_hid,
            "gates": mask_gates,
            "z_input": mask_in,
            "z_hidden_by_w": mask_hid_by_w,
            "z_hidden": mask_hid_by_z,
        }
    
    def forward(self, inputs, hid_init=None, deterministic: bool = False, clip: bool = False, **kwargs):
        """
        PyTorch forward for BayesianLSTM — повторяет логику get_output_for.
        inputs: либо тензор (batch, seq_len, input_dim) либо список/tuple:
                inputs[0] - input tensor,
                при self.mask_incoming_index > 0 mask ожидается в inputs[self.mask_incoming_index],
                при self.hid_prop ожидается inputs[1] с hid_init и cell_init.
        Возвращает:
        - если self.only_return_final: (batch, num_units)
        - elif self.hid_prop: (2, batch, seq_len, num_units)
        - else: (batch, seq_len, num_units)
        """

        # --- извлекаем input и mask как в оригинале ---
        if isinstance(inputs, (list, tuple)):
            x = inputs[0]
        else:
            x = inputs

        mask = None
        if getattr(self, "mask_incoming_index", -1) > 0 and isinstance(inputs, (list, tuple)):
            # защитимся от выхода за пределы
            if len(inputs) > self.mask_incoming_index:
                mask = inputs[self.mask_incoming_index]

        # ожидаем вход (batch, seq_len, input_dim) -> переставляем (seq_len, batch, input_dim)
        x = x.permute(1, 0, 2)
        seq_len, num_batch, _ = x.shape
        dtype = torch.float32

        # --- подготовка шума/клип-масок (перенесём результаты на нужное device/dtype) ---
        self.generate_noise_and_clip(num_batch, deterministic, clip)

        # gates_noise может иметь форму (4, num_batch, num_units) или (4, num_units) или (4,)
        # приведём g* к форме (batch, num_units) через broadcast
        if self.gates_noise is not None:
            g0 = self.gates_noise[0]
            g1 = self.gates_noise[1]
            g2 = self.gates_noise[2]
            g3 = self.gates_noise[3]
            # если нужно, расширим dim для batch
            if g0.ndim == 1:
                g0 = g0.unsqueeze(0).expand(num_batch, -1)
                g1 = g1.unsqueeze(0).expand(num_batch, -1)
                g2 = g2.unsqueeze(0).expand(num_batch, -1)
                g3 = g3.unsqueeze(0).expand(num_batch, -1)
            elif g0.ndim == 2 and g0.shape[0] == num_batch:
                pass  # уже (batch, units)
            else:
                # если gates_noise имелось в форме (4, batch, units), индексирование уже верно
                g0 = g0 if g0.ndim == 2 else g0.to(dtype=dtype)
                g1 = g1 if g1.ndim == 2 else g1.to(dtype=dtype)
                g2 = g2 if g2.ndim == 2 else g2.to(dtype=dtype)
                g3 = g3 if g3.ndim == 2 else g3.to(dtype=dtype)
        else:
            g0 = g1 = g2 = g3 = torch.ones((num_batch, self.num_units), dtype=dtype)
        
        # gates_clip can be (4, num_units) or (4,) or tensor
        def _gclip(idx):
            if self.gates_clip is None:
                return torch.ones((self.num_units,), dtype=dtype)
            v = self.gates_clip[idx]
            if v.ndim == 0:
                return v.expand(self.num_units).to(dtype=dtype)
            return v.to(dtype=dtype)
        gc0 = _gclip(0)
        gc1 = _gclip(1)
        gc2 = _gclip(2)
        gc3 = _gclip(3)
        
        g0_gc0 = g0 * gc0
        g1_gc1 = g1 * gc1
        g2_gc2 = g2 * gc2
        g3_gc3 = g3 * gc3
    
        # --- предактивации входной части (seq_len, batch, num_units) ---
        input_i = self.input_preactivation(x, 'input', deterministic=deterministic, clip=clip, **kwargs) + self.b_ingate
        input_f = self.input_preactivation(x, 'forget', deterministic=deterministic, clip=clip, **kwargs) + self.b_forgetgate
        input_c = self.input_preactivation(x, 'cell', deterministic=deterministic, clip=clip, **kwargs) + self.b_cell
        input_o = self.input_preactivation(x, 'output', deterministic=deterministic, clip=clip, **kwargs) + self.b_outgate

        if mask is not None:
            if mask.ndim == 3:
                mask_seq = mask.permute(1, 0, 2).to(dtype=dtype)
            else:
                mask_seq = mask.unsqueeze(-1).permute(1, 0, 2).to(dtype=dtype)
        else:
            mask_seq = None
        
        # --- начальные состояния ---
        if self.hid_prop and hid_init is not None:
            hid = hid_init[0].to(dtype=dtype)
            cell = hid_init[1].to(dtype=dtype)
        else:
            hid = None
            cell = None

        hid_out = torch.empty((seq_len, num_batch, self.num_units), dtype=self.dtype)
        cell_out = torch.empty_like(hid_out)

        # перебор по времени (учёт backwards)
        t_range = range(seq_len - 1, -1, -1) if self.backwards else range(seq_len)

        for t in t_range:
            cell_prev = cell
            hid_prev = hid
            
            # предактивации для текущего шага (batch, num_units)
            input_n_i = input_i[t]
            input_n_f = input_f[t]
            input_n_c = input_c[t]
            input_n_o = input_o[t]

            # рекуррентные предактивации
            hid_preact_i = self.hidden_preactivation(hid, 'input', deterministic=deterministic, clip=clip, **kwargs)
            hid_preact_f = self.hidden_preactivation(hid, 'forget', deterministic=deterministic, clip=clip, **kwargs)
            hid_preact_c = self.hidden_preactivation(hid, 'cell', deterministic=deterministic, clip=clip, **kwargs)
            hid_preact_o = self.hidden_preactivation(hid, 'output', deterministic=deterministic, clip=clip, **kwargs)

            # вычисление гейтов согласно оригиналу
            ingate = self.nonlinearity_ingate((input_n_i + hid_preact_i) * g0_gc0 + self.b_ingate)
            forgetgate = self.nonlinearity_forgetgate((input_n_f + hid_preact_f) * g1_gc1 + self.b_forgetgate)
            cell_candidate = self.nonlinearity_cell((input_n_c + hid_preact_c) * g2_gc2 + self.b_cell)
            
            cell_new = forgetgate * cell_prev + ingate * cell_candidate
            outgate = self.nonlinearity_outgate((input_n_o + hid_preact_o) * g3_gc3 + self.b_outgate)
            hid_new = outgate * self.nonlinearity(cell_new)

            # применение шума/клипа к скрытому
            hn = self.hidden_noise if self.hidden_noise is not None else torch.ones(1, dtype=dtype)
            hc = self.hidden_clip if self.hidden_clip is not None else torch.ones(1, dtype=dtype)
            hid_new = hid_new * hn * hc

            # маскирование, если есть mask
            if mask_seq is not None:
                m = mask_seq[t]  # (batch, 1) или (batch, units) в редком случае
                # приводим к broadcastable форме (batch, units)
                if m.ndim == 2 and m.shape[1] == 1:
                    m = m.expand(-1, self.num_units)
                cell = m * cell_new + (1.0 - m) * cell_prev
                hid = m * hid_new + (1.0 - m) * hid_prev
            else:
                cell = cell_new
                hid = hid_new

            hid_out[t] = hid
            cell_out[t] = cell

        # only_return_final
        if self.only_return_final:
            return hid_out[-1]

        # восстановление порядка времени, если нужно
        if self.backwards:
            hid_out = hid_out.flip(dims=[0])
            cell_out = cell_out.flip(dims=[0])

        # в (batch, seq_len, num_units)
        hid_out = hid_out.permute(1, 0, 2)
        cell_out = cell_out.permute(1, 0, 2)

        if self.hid_prop:
            return torch.cat([hid_out.unsqueeze(0), cell_out.unsqueeze(0)], dim=0)
        else:
            return hid_out



In [ ]:
class Dense(nn.Module):
    def __init__(self, incoming, num_units, nonlinearity=nn.Identity()):
        super(Dense, self).__init__()
        self.num_units = num_units
        self.nonlinearity = nonlinearity
        
        self.W = nn.Parameter(torch.empty(incoming, num_units))
        self.b = nn.Parameter(torch.zeros(num_units))
        
        nn.init.xavier_uniform_(self.W)

    def pre_activation(self, input):
        return torch.matmul(input, self.W)
    
    def get_output_shape_for(self, input_shape):
        return tuple(input_shape[:-1]) + (self.num_units,)

    def get_output_for(self, input, **kwargs):
        return self.nonlinearity(self.pre_activation(input, **kwargs) + self.b)

    def get_ard(self):
        return {"w": torch.ones_like(self.W)}
    
    def forward(self, input, **kwargs):
        """
        input: tensor with last dim == incoming, arbitrary leading dims allowed
        returns: tensor with same leading dims and last dim == num_units
        """
        # pre-activation: (..., incoming) @ (incoming, num_units) -> (..., num_units)
        lin = self.pre_activation(input)
        # add bias (1D) -> broadcasting over leading dims
        lin = lin + self.b
        # apply nonlinearity (can be nn.Module or a callable)
        return self.nonlinearity(lin)

class BayesianDense(Dense):
    def __init__(self, 
                 incoming, 
                 num_units, 
                 log_sigma_init = -3.0,
                 W_initializer=None, 
                 b_init=0.0, 
                 nonlinearity=lambda x: x, 
                 **kwargs):
        super().__init__(incoming, num_units, nonlinearity, **kwargs)

        if isinstance(incoming, int):
            self.num_inputs = int(incoming)
        else:
            try:
                self.num_inputs = int(incoming[-1])
            except Exception:
                raise ValueError("incoming must be int or shape-like")

        self.num_units = int(num_units)
        self.nonlinearity = nonlinearity
        self.name = 'DenseSparseVDO'
        self.thresh = 3.0

        # веса и bias (аналог add_param)
        self.W = nn.Parameter(torch.empty(self.num_inputs, self.num_units, dtype=torch.float32))
        self.b = nn.Parameter(torch.full((self.num_units,), float(b_init), dtype=torch.float32))
        # log_sigma (trainable)
        self.log_sigma = nn.Parameter(torch.full((self.num_inputs, self.num_units), float(log_sigma_init), dtype=torch.float32))

        # инициализация W (GlorotUniform по умолчанию)
        if W_initializer is None:
            nn.init.xavier_uniform_(self.W)
        else:
            # попытаться применить инициализатор
            try:
                W_initializer(self.W)
            except Exception:
                # если инициализатор возвращает numpy
                self.W.data.copy_(torch.tensor(W_initializer(self.W.shape), dtype=self.W.dtype))

        # генератор случайных чисел (аналог RandomStreams)
        seed = np.random.randint(int(1e7))
        self._srng = torch.Generator()
        self._srng.manual_seed(int(seed))

    def clip_func(self, mtx: torch.Tensor, to: float = 8.0) -> torch.Tensor:
        return torch.clamp(mtx, min=-float(to), max=float(to))

    def pre_activation(self, input: torch.Tensor, deterministic: bool = False, clip: bool = False):
        """
        input: либо 2D (batch, input_dim) либо 3D (batch, seq_len, input_dim)
        Возвращает: mu + шум*si (или только mu в deterministic режиме)
        Сохраняет логику оригинала (шум shape для 3D -> (batch,1,num_units))
        """
        device = self.W.device
        dtype = self.W.dtype

        if deterministic:
            if clip:
                # log_alpha = clip_func(2*self.log_sigma - log(self.W ** 2))
                log_alpha = utils.clip_func(2.0 * self.log_sigma - utils.safe_torch_log(self.W.pow(2)))
                clip_mask = log_alpha.ge(self.thresh)  # True где нужно обрезать
                W_eff = torch.where(clip_mask, torch.zeros_like(self.W), self.W)
                return torch.matmul(input, W_eff)
            else:
                return torch.matmul(input, self.W)

        # стохастический режим
        W = self.W
        sigma2 = torch.exp(2.0 * self.log_sigma)

        if clip:
            log_alpha = utils.clip_func(2.0 * self.log_sigma - utils.safe_torch_log(self.W.pow(2)))
            clip_mask = log_alpha.ge(self.thresh)
            W = torch.where(clip_mask, torch.zeros_like(self.W), self.W)
            # зануляем sigma2 для обрезанных весов (как в Theano)
            sigma2 = torch.where(clip_mask, torch.zeros_like(sigma2), sigma2)

        # mu = input @ W
        mu = torch.matmul(input, W)  # broadcasting работает для 2D/3D: (..., in) @ (in, out) -> (..., out)

        # si = sqrt( (input*input) @ sigma2 + eps )
        si = torch.sqrt(torch.matmul(input * input, sigma2) + 1e-8)  # shape matches mu

        # создаём шум как в Theano:
        if input.ndim == 2:
            # input shape (batch, in) -> mu shape (batch, out)
            noise = torch.randn(mu.shape, generator=self._srng, device=device, dtype=dtype)
            return mu + noise * si
        else:
            # input shape (batch, seq_len, in) -> mu shape (batch, seq_len, out)
            # Theano генерировал noise shape (batch, 1, out) — один шум на все временные шаги
            noise = torch.randn((mu.shape[0], 1, mu.shape[2]), generator=self._srng, device=device, dtype=dtype)
            return mu + noise * si

    def eval_reg(self, train_size: float):
        """
        alpha regularization: utils.alpha_regf(clip_func(2*log_sigma - log(W^2))).sum() / train_size
        Возвращаем torch scalar
        """
        log_alpha = utils.clip_func(2.0 * self.log_sigma - utils.safe_torch_log(self.W.pow(2)))
        reg = utils.alpha_regf(log_alpha).sum() / float(train_size)
        # print(f"    BayesianDense.evel_reg reg: {reg}")
        return reg

    def get_ard(self) -> dict[str, torch.Tensor]:
        """
        Возвращаем torch-маску (bool tensor).
        Маска не требует градиентов и используется для sparsification.
        """
        W = self.W.detach()
        log_sigma = self.log_sigma.detach()
        log_alpha = 2.0 * log_sigma - 2.0 * utils.safe_torch_log(torch.abs(W))
        mask = (log_alpha < self.thresh)

        return {"w": mask}
    
    def forward(self, input: torch.Tensor, deterministic: bool = False, clip: bool = False, **kwargs):
        """
        input: tensor with last dim == num_inputs, can be 2D (batch, in) or 3D (batch, seq_len, in)
        deterministic, clip: передаются в pre_activation и управляют режимом (как в Theano-оригинале)
        Возвращает: nonlinearity( pre_activation(input, deterministic, clip) + b )
        """
        # pre_activation вернёт mu + noise*si (или просто mu в deterministic режиме)
        out = self.pre_activation(input, deterministic=deterministic, clip=clip)

        # прибавляем bias; bias shape (num_units,) автоматически broadcast'ится по всем leading dims
        out = out + self.b

        # применяем nonlinearity — может быть nn.Module или простая функция
        return self.nonlinearity(out)


class BayesianDense_noLRT(Dense):
    def __init__(self, incoming, num_units, log_sigma_init=-3.0,
                 W_initializer=None, b_init=0.0, nonlinearity=lambda x: x, **kwargs):
        """
        incoming: int (input size) or shape-like with last dim = input size.
        """
        super().__init__(incoming, num_units, nonlinearity, **kwargs)
        # определить num_inputs как в Dense
        if isinstance(incoming, int):
            self.num_inputs = int(incoming)
        else:
            try:
                self.num_inputs = int(incoming[-1])
            except Exception:
                raise ValueError("incoming must be int or shape-like")

        self.num_units = int(num_units)
        self.nonlinearity = nonlinearity
        self.name = 'DenseSparseVDO'
        self.thresh = 3.0

        # параметры: W, b, log_sigma
        self.W = nn.Parameter(torch.empty(self.num_inputs, self.num_units, dtype=torch.float32))
        self.b = nn.Parameter(torch.full((self.num_units,), float(b_init), dtype=torch.float32))
        self.log_sigma = nn.Parameter(torch.full((self.num_inputs, self.num_units), float(log_sigma_init), dtype=torch.float32))

        # инициализация весов (GlorotUniform ≈ xavier_uniform)
        if W_initializer is None:
            nn.init.xavier_uniform_(self.W)
        else:
            try:
                W_initializer(self.W)
            except Exception:
                self.W.data.copy_(torch.tensor(W_initializer(self.W.shape), dtype=self.W.dtype))

        # генератор для повторяемых нормальных выборок (аналог RandomStreams)
        seed = np.random.randint(int(1e7))
        self._srng = torch.Generator()
        self._srng.manual_seed(int(seed))

    def clip_func(self, mtx: torch.Tensor, to: float = 8.0) -> torch.Tensor:
        return torch.clamp(mtx, min=-float(to), max=float(to))

    def pre_activation(self, input: torch.Tensor, deterministic: bool = False, clip: bool = False):
        """
        input: 2D (batch, in) или 3D (batch, seq_len, in) (и др. формы с последним измерением in)
        Ведёт себя как оригинал Theano-кода (с исправлением очевидных опечаток в использовании clip_train/self.clip).
        """
        device = self.W.device
        dtype = self.W.dtype

        # детерминированный режим
        if deterministic:
            if clip:
                log_alpha = utils.clip_func(2.0 * self.log_sigma - utils.safe_torch_log(self.W.pow(2)))
                clip_mask = log_alpha.ge(self.thresh)  # True где надо обрезать
                W_eff = torch.where(clip_mask, torch.zeros_like(self.W), self.W)
                return torch.matmul(input, W_eff)
            else:
                return torch.matmul(input, self.W)

        # стохастический режим
        # поведение разделяется по размерности входа:
        if input.ndim == 2:
            # Вариационный шум на выходе (input-dependent std)
            W = self.W
            sigma2 = torch.exp(2.0 * self.log_sigma)  # var = exp(2*log_sigma)

            # в оригинале была опечатка clip_train — используем clip аргумент
            if clip:
                log_alpha = utils.clip_func(2.0 * self.log_sigma - utils.safe_torch_log(self.W.pow(2)))
                clip_mask = log_alpha.ge(self.thresh)
                W = torch.where(clip_mask, torch.zeros_like(self.W), W)
                sigma2 = torch.where(clip_mask, torch.zeros_like(sigma2), sigma2)

            mu = torch.matmul(input, W)  # (batch, out)
            si = torch.sqrt(torch.matmul(input * input, sigma2) + 1e-8)  # (batch, out)

            noise = torch.randn(mu.shape, generator=self._srng, device=device, dtype=dtype)
            return mu + noise * si

        else:
            # input.ndim != 2 -> sample noisy weights once and apply to full input (e.g., sequence)
            # W_noisy = W + N(0,1) * exp(log_sigma)
            W_noise = torch.randn(self.W.shape, generator=self._srng, device=device, dtype=dtype) * torch.exp(self.log_sigma)
            W_noisy = self.W + W_noise

            if clip:
                log_alpha = utils.clip_func(2.0 * self.log_sigma - utils.safe_torch_log(self.W.pow(2)))
                clip_mask = log_alpha.ge(self.thresh)
                W_noisy = torch.where(clip_mask, torch.zeros_like(W_noisy), W_noisy)

            # input shape (batch, seq_len, in), W_noisy (in, out) -> output (batch, seq_len, out)
            return torch.matmul(input, W_noisy)

    def eval_reg(self, train_size: float):
        eps = 1e-8
        log_alpha = utils.clip_func(2.0 * self.log_sigma - utils.safe_torch_log(self.W.pow(2)))
        reg = utils.alpha_regf(log_alpha).sum() / float(train_size)
        # print(f"    BayesianDense_noLRT.eval_reg reg: {reg}")
        return reg
    
    def get_ard(self) -> dict[str, torch.Tensor]:
        """
        Возвращаем torch-маску (bool tensor).
        Маска не требует градиентов и используется для sparsification.
        """
        eps = 1e-8
        W = self.W.detach()
        log_sigma = self.log_sigma.detach()
        log_alpha = 2.0 * log_sigma - 2.0 * utils.safe_torch_log(torch.abs(W))
        mask = (log_alpha < self.thresh)
        return {"w": mask}
    
    def forward(self, input: torch.Tensor, deterministic: bool = None, clip: bool = False, **kwargs) -> torch.Tensor:
        """
        input: tensor with last dim == num_inputs, can be 2D (batch, in) or 3D (batch, seq_len, in)
        deterministic: if None -> deterministic = not self.training (eval mode); else use explicit value
        clip: whether to apply clipping logic inside pre_activation
        """
        # по умолчанию привязываем deterministic к режиму модуля,
        # это удобно: model.eval() -> deterministic=True, model.train() -> deterministic=False
        if deterministic is None:
            deterministic = not self.training

        # получаем предактивацию (mu + noise*si или просто mu в deterministic режиме)
        out = self.pre_activation(input, deterministic=deterministic, clip=clip)

        # добавляем bias (broadcast по всем ведущим осям)
        out = out + self.b

        # применяем nonlinearity (может быть nn.Module или callable)
        return self.nonlinearity(out)

In [18]:
class LMNet(nn.Module):
    def __init__(
            self, 
            vocab_size, 
            n_hidden, 
            config, 
            hid_prop=False, 
            batch_size=32,
            device='cpu',
            get_true_probs=False):
        """
        vocab_size: размер словаря
        n_hidden: размер скрытого состояния LSTM
        config: строка конфигурации (как в оригинале)
        hid_prop: если True — предусмотрена возможность прокинуть hid_init
        batch_size: использовался в оригинале при создании shared hid (32)
        device: 'cpu' или 'cuda'
        """
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.n_hidden = int(n_hidden)
        self.config = config
        self.hid_prop = bool(hid_prop)
        self.batch_size = int(batch_size)
        self.device = device
        self.get_true_probs = get_true_probs

        # LSTM: принимает вход feature dim = vocab_size
        # Передаём конфигурацию так, чтобы LSTM использовал те же настройки и инициализации
        self.lstm = BayesianLSTM(
            incoming=self.vocab_size,
            num_units=self.n_hidden,
            config=config[:3],
            only_return_final=False,
            learn_init=False,
            hid_prop=hid_prop
            # В твоём портированном BayesianLSTM можно добавить аргументы для инициализации ворот,
            # чтобы сохранить Orthogonal(...) и т.д.
        )

        # hid buffer — аналог theano.shared(np.zeros((2,32, n_hidden)))
        # сохраняем в качестве buffer чтобы было в state_dict, но не обучаемо
        if self.hid_prop:
            hid_tensor = torch.zeros((2, self.batch_size, self.n_hidden), dtype=torch.float32)
            # как в Theano, hid инициализируется нулями; в PyTorch сделаем buffer
            self.register_buffer('hid', hid_tensor)

        # dense: если hid_prop, в оригинале Dense применялся к определенному срезу LSTM-выхода;
        # здесь реализуем Dense, который можно вызвать на тензоре формы (batch, seq_len, n_hidden).
        if self.hid_prop:
            # В оригинале они брали SliceLayer(..., indices=0, axis=0) — в итоге Dense применялся по времени.
            # В PyTorch просто создаём Dense, который применяют к LSTM-выходу по временным шагам.
            DenseClass = BayesianDense_noLRT if config[-1] == "L" else Dense
            self.dense = DenseClass(incoming=self.n_hidden, num_units=self.vocab_size)
            # to_init_lstm в оригинале получали срез для инициализации LSTM в следующей итерации:
            # здесь мы будем извлекать последний скрытый (по времени) в forward: hid_to_init = hid_out[:, -1, :]
        else:
            DenseClass = BayesianDense if config[-1] == "L" else Dense
            self.dense = DenseClass(incoming=self.n_hidden, num_units=self.vocab_size)

        # softmax на последнем измерении; в оригинале они делали Reshape -> softmax,
        # мы сделаем то же в forward (flatten batch*seq, dim vocab) -> F.softmax
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, inp, use_hid_init=False):
        """
        inp: LongTensor (batch, seq_len) — индексы слов (как T.imatrix)
        use_hid_init: если True и hid_prop=True — использовать self.hid как hid_init
        Возвращает: распределения (batch*seq_len, vocab_size) или (batch, seq_len, vocab_size) в зависимости от нужд.
        """
        
        if self.hid_prop:
            b = inp.size(0)
            hid_init = self.hid_next[:, :b, :].clone() if hasattr(self, 'hid_next') else self.hid[:, :b, :].clone()
            lstm_out = self.lstm(inp, hid_init=hid_init)
            
            # last_hidden по последнему времени, detach чтобы не ломать граф градиента
            last_hidden = lstm_out[:, :, -1, :].detach()
            
            # сохраняем для следующего батча, не мутируя граф
            self.hid_next = last_hidden.clone()
        else:
            lstm_out = self.lstm(inp)

        # dense должен уметь принимать (..., in) тензор и вернуть (..., out)
        hid_out = lstm_out[0]  # (batch, seq_len, n_hidden)
        logits = self.dense(hid_out)  # (batch, seq_len, vocab_size)

        # 5) flatten (batch*seq_len, vocab) and softmax (как в оригинале Reshape + NonlinearityLayer(softmax))
        if self.get_true_probs:
            b, s, v = logits.shape
            logits_flat = logits.contiguous().view(-1, v)
            probs_flat = self.softmax(logits_flat)
            probs = probs_flat.view(b, s, v)
        else:
            probs = 0
            
        return probs, logits

    # --- перенос compute_compression_masks и evaluate_compression, почти без изменений логики ---
    def compute_compression_masks(self):
        """
        Возвращает маски компрессии в виде torch.Tensor (на device модели).
        Маски отвязаны от графа вычислений (detach).
        """

        device = next(self.parameters()).device

        # --- получаем маски ARD из слоёв (уже torch.Tensor из get_ard) ---
        masks_lstm = self.lstm.get_ard()    # dict[str, torch.Tensor]
        masks_dense = self.dense.get_ard()  # dict[str, torch.Tensor]

        # --- агрегируем маски ---
        mask_vocabulary = masks_lstm["z_input"].to(dtype=torch.bool, device=device)

        mask_hidden = torch.logical_or(
            masks_lstm["z_hidden_by_w"],
            masks_dense["w"].any(dim=1)
        )
        mask_hidden = torch.logical_and(mask_hidden, masks_lstm["z_hidden"])
        mask_hidden = mask_hidden.to(dtype=torch.bool, device=device)

        # --- клонируем веса (чтобы не портить оригинальные маски) ---
        w_in = masks_lstm["w_input"].clone().to(device)
        w_hid = masks_lstm["w_hidden"].clone().to(device)
        w_dense = masks_dense["w"].clone().to(device)
        gates = masks_lstm["gates"].clone().to(device)

        # --- применяем маски ---
        w_in[:, ~mask_vocabulary, :] = 0
        w_in[:, :, ~mask_hidden] = 0
        w_hid[:, ~mask_hidden, :] = 0
        w_hid[:, :, ~mask_hidden] = 0
        w_dense[~mask_hidden] = 0
        gates[:, ~mask_hidden] = 0

        return mask_vocabulary, mask_hidden, gates, w_in, w_hid, w_dense

    def evaluate_compression(self):
        mask_vocabulary, mask_hidden, mask_gates, mask_w_inp, mask_w_hid, mask_w_dense = self.compute_compression_masks()
        # print(f"mask_vocabulary: {mask_vocabulary}")
        # print(f"mask_hidden: {mask_hidden}")
        # print(f"mask_gates: {mask_gates}")
        # print(f"mask_w_inp: {mask_w_inp}")
        # print(f"mask_w_hid: {mask_w_hid}")
        # print(f"mask_w_dense: {mask_w_dense}")
        
        # compute overall compression
        w_nonzero, w_all = 0.0, 0.0
        for w in [mask_w_inp, mask_w_hid, mask_w_dense]:
            w_nonzero += float(w.sum())
            w_all += float(w.numel())
        overall_compression = w_all / (w_nonzero + 1e-8)

        # print compression per layer (как в оригинале)
        print("Compression per layers:")
        for layer_name, masks in [("LSTM", {"z_h":mask_hidden, "gates":mask_gates, "w_h":mask_w_hid, "w_x":mask_w_inp, "z_x":mask_vocabulary}), ("Dense", {"w":mask_w_dense})]:
            print(layer_name, end=": ")
            for key, matrix in masks.items():
                print("(%s: %d/%d)" % (key, int(matrix.sum()), matrix.numel()), end=" ")
            print()
        print("Overall compression:", overall_compression)
        return overall_compression

In [27]:
def train_char_lm_model(
    model, 
    train_loader, 
    test_loader, 
    optimizer, 
    criterion,
    num_epochs, 
    valid_loader=None,
    grad_clip=10.0, 
    print_fq=1, 
    save_fq=0, 
    file_name="model.pt",
    sparsification_eval_fun=None, 
    device="cpu"
    ):
    
    print("Training ...")

    num_batches = len(train_loader)
    train_size = train_loader.train_size()
    
    for epoch in range(1, num_epochs + 1):
        start = time.perf_counter()
        model.train()
        total_loss = 0.0
        total_tokens = 0
        grad_norm = 0.0

        for batch_idx, (x, y) in enumerate(train_loader):  # PTBWordLoader → torch.Tensor
            optimizer.zero_grad()
            
            probs, logits = model(x)
            reg = sum([layer.eval_reg(train_size)
                    for i, layer in enumerate(model.modules())
                    if hasattr(layer, "eval_reg")])
            # print(f"train_char_lm_model logits: {logits}")
                        
            base_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss = base_loss + reg

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            loss_item = loss.item()
            y_numel = y.numel()
            
            total_loss += loss_item * y_numel
            total_tokens += y_numel
            print(f"epoch: {epoch}/{num_epochs}; "
                  f"batch: {batch_idx}/{num_batches}; "
                  f"loss: {loss_item:.4f}; tokens: {y_numel}")
            # для мониторинга нормы градиента
            grad_norm = max(grad_norm, torch.norm(
                torch.stack([p.grad.norm() for p in model.parameters() if p.grad is not None])
            ).item())

        avg_train_loss = total_loss / total_tokens

        # ----- Валидация -----
        if epoch % print_fq == 0 or epoch == num_epochs:
            def evaluate(loader):
                model.eval()
                loss_total, tokens_total = 0.0, 0
                with torch.no_grad():
                    for x, y in loader:
                        probs, logits = model(x)
                        base_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
                        reg = sum([layer.eval_reg(train_size)
                            for i, layer in enumerate(model.modules())
                            if hasattr(layer, "eval_reg")])
                        loss = base_loss + reg
                        loss_total += loss.item() * y.numel()
                        tokens_total += y.numel()
                return loss_total / tokens_total

            val_loss = evaluate(valid_loader) if valid_loader else None
            test_loss = evaluate(test_loader)
            end = time.perf_counter()
            print(f"Epoch {epoch:03d} | Tooks {end - start:.2f}s | "
                  f"Train loss {avg_train_loss:.4f} | "
                  f"Grad norm {grad_norm:.4f} | "
                  f"Val loss {val_loss:.4f} | Test loss {test_loss:.4f}")

            if sparsification_eval_fun:
                sparsification_eval_fun()

        # ----- Сохранение -----
        if save_fq and epoch % save_fq == 0:
            torch.save(model.state_dict(), f"{file_name}_epoch_{epoch}.pt")

    return model



In [21]:
# from typing import Optional, Callable, Dict, Any, Iterable


# class LMTrainer:
#     """
#     Тренировочный helper для LMNet / BayesianLSTM pipeline.
#     - model: ваша модель (LMNet или совместимая)
#     - optimizer: torch optimizer
#     - device: 'cpu' или 'cuda'
#     - loss_fn: функция потерь, ожидает logits (N, C) и targets (N,) — по умолчанию CrossEntropyLoss
#     - val_loss_fn: аналогично (defaults to loss_fn)
#     - grad_clip: значение clip (L2 norm)
#     - mc_samples: сколько прогонов для MC-оценки
#     - test_types: список режимов валидации, например ["MC", "MC_clip", "usual", "clip"]
#     - input_is_indices: если True — входные x считаются индексами и автоматически преобразуются в one-hot
#       (если модель имеет атрибут vocab_size, используется он).
#     - train_size: используем для регуляризатора (если None — попытаемся вывести из train_data)
#     - hid_prop: модель поддерживает прокидку hid между батчами (если да — trainer поддерживает reset_hid())
#     """
#     def __init__(self,
#                  model: nn.Module,
#                  optimizer: torch.optim.Optimizer,
#                  device: str = "cpu",
#                  loss_fn: Optional[Callable] = None,
#                  val_loss_fn: Optional[Callable] = None,
#                  grad_clip: float = 10.0,
#                  mc_samples: int = 10,
#                  test_types: Optional[Iterable[str]] = None,
#                  input_is_indices: bool = False,
#                  train_size: Optional[float] = None,
#                  hid_prop: bool = False):
#         self.device = torch.device(device)
#         self.model = model.to(self.device)
#         self.optimizer = optimizer
#         self.loss_fn = nn.CrossEntropyLoss(reduction="mean") if loss_fn is None else loss_fn
#         self.val_loss_fn = self.loss_fn if val_loss_fn is None else val_loss_fn
#         self.grad_clip = grad_clip
#         self.mc_samples = int(mc_samples)
#         self.test_types = list(test_types) if test_types is not None else ["MC", "MC_clip", "usual", "clip"]
#         self.input_is_indices = bool(input_is_indices)
#         self.train_size = train_size
#         self.hid_prop = bool(hid_prop)

#         # try to obtain vocab_size from model if possible
#         self.vocab_size = getattr(model, "vocab_size", None)

#     # ---------- internal helpers ----------
#     def _to_tensor(self, x, dtype=None):
#         if torch.is_tensor(x):
#             t = x.to(device=self.device)
#         else:
#             t = torch.as_tensor(x, device=self.device)
#         if dtype is not None:
#             t = t.to(dtype=dtype)
#         return t

#     def _compute_reg(self):
#         # Sum eval_reg over modules that have such method
#         regs = []
#         for m in self.model.modules():
#             if hasattr(m, "eval_reg"):
#                 r = m.eval_reg(self.train_size if self.train_size is not None else 1.0)
#                 # r can be python float or torch tensor
#                 if not torch.is_tensor(r):
#                     r = torch.tensor(r, device=self.device, dtype=torch.float32)
#                 else:
#                     r = r.to(device=self.device)
#                 regs.append(r)
#         if len(regs) == 0:
#             return torch.tensor(0., device=self.device)
#         return torch.stack(regs).sum()

#     @staticmethod
#     def _total_grad_norm(params):
#         tot = 0.0
#         for p in params:
#             if p.grad is None:
#                 continue
#             param_norm = p.grad.detach().data.norm(2)
#             tot += (param_norm.item() ** 2)
#         return float(tot ** 0.5)

#     # ---------- forward wrappers ----------
#     def _model_forward(self, x, deterministic=False, clip=False, hid_init=None):
#         """
#         Calls model; supports models that return:
#            - logits
#            - (probs, logits)
#            - for hid_prop case model may set model.hid_next
#         Returns (logits_tensor, probs_tensor_or_None)
#         """
#         # attempt call with kwargs if model supports them
#         try:
#             out = self.model(x, deterministic=deterministic, clip=clip, hid_init=hid_init)
#         except TypeError:
#             out = self.model(x)

#         if isinstance(out, (tuple, list)):
#             if len(out) == 2:
#                 probs, logits = out
#             else:
#                 logits = out[-1]
#                 probs = out[-2] if len(out) >= 2 else None
#         else:
#             logits = out
#             probs = None

#         return logits, probs

#     # ---------- train / eval step ----------
#     def train_step(self, x, y):
#         """
#         One training step. x_raw and y_raw can be numpy or torch tensors.
#         Returns (loss_value: float, grad_norm_before_clipping: float)
#         """

#         self.model.train()
#         self.optimizer.zero_grad()

#         logits, probs = self._model_forward(x, deterministic=False, clip=False)
#         # logits expected shape (batch, seq_len, vocab)
#         if logits is None:
#             raise RuntimeError("Model returned None logits in train_step.")

#         # flatten and compute loss
#         b, s, C = logits.shape
#         logits_flat = logits.contiguous().view(-1, C)
#         targets_flat = y.contiguous().view(-1)
#         base_loss = self.loss_fn(logits_flat, targets_flat)

#         reg = self._compute_reg()
#         loss = base_loss + reg

#         loss.backward()

#         total_norm = self._total_grad_norm([p for p in self.model.parameters() if p.requires_grad])
#         if self.grad_clip is not None and self.grad_clip > 0:
#             torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
#         self.optimizer.step()

#         # update hid buffer if model supports hid_prop via model.hid_next
#         if self.hid_prop:
#             if hasattr(self.model, "hid_next") and getattr(self.model, "hid_next") is not None:
#                 with torch.no_grad():
#                     hnext = self.model.hid_next.detach()
#                     if hasattr(self.model, "hid"):
#                         cur_b = hnext.size(1)
#                         # write into buffer
#                         self.model.hid[:, :cur_b, :] = hnext.to(self.model.hid.device)
#                     else:
#                         # fallback: keep as attribute
#                         self.model.hid = hnext.detach().clone()

#         # detect NaN and return
#         loss_val = float(loss.item())
#         if np.isnan(loss_val):
#             # Helpful debug printouts
#             print("=== NaN detected in loss ===")
#             try:
#                 print("logits stats:", torch.nanmean(logits).item(), torch.nanstd(logits).item())
#             except Exception:
#                 pass
#             # Print some parameter norms
#             for name, p in self.model.named_parameters():
#                 if p.grad is not None:
#                     print(f"param {name} norm: {p.data.norm().item():.6f}, grad norm: {p.grad.data.norm().item():.6f}")
#             raise FloatingPointError("NaN in loss (train_step).")
#         return loss_val, float(total_norm)

#     def eval_batch(self, x_raw, y_raw, mode="usual"):
#         """Evaluate single batch. mode in {'usual','clip','MC','MC_clip'}"""
#         clip_flag = ("clip" in mode)
#         mc_flag = ("MC" in mode)

#         # prepare inputs like in train_step
#         if self.input_is_indices:
#             x_t = self._to_tensor(x_raw, dtype=torch.long)
#             x = self._maybe_make_onehot(x_t)
#         else:
#             if (torch.is_tensor(x_raw) and x_raw.dtype in (torch.int32, torch.int64)) or (isinstance(x_raw, np.ndarray) and np.issubdtype(x_raw.dtype, np.integer)):
#                 x_t = self._to_tensor(x_raw, dtype=torch.long)
#                 x = self._maybe_make_onehot(x_t)
#             else:
#                 x = self._to_tensor(x_raw, dtype=None).float()

#         y = self._to_tensor(y_raw, dtype=torch.long)
#         self.model.eval()

#         with torch.no_grad():
#             if mc_flag:
#                 # average probabilities across mc_samples
#                 probs_acc = None
#                 C = None
#                 for i in range(self.mc_samples):
#                     logits_i, probs_i = self._model_forward(x, deterministic=False, clip=clip_flag)
#                     if probs_i is None:
#                         probs_i = nn.functional.softmax(logits_i, dim=-1)
#                     if probs_acc is None:
#                         probs_acc = probs_i.detach().clone()
#                     else:
#                         probs_acc = probs_acc + probs_i.detach()
#                     if C is None:
#                         C = probs_i.size(-1)
#                 p_avg = probs_acc / float(self.mc_samples)
#                 p_avg_flat = p_avg.contiguous().view(-1, C)
#                 targets_flat = y.contiguous().view(-1)
#                 # negative log prob mean
#                 eps = 1e-12
#                 p_sel = p_avg_flat[torch.arange(p_avg_flat.size(0), device=p_avg_flat.device), targets_flat].clamp(min=eps)
#                 loss = (-torch.log(p_sel)).mean().item()
#                 # Also update hid if model produced hid_next in last run
#                 if self.hid_prop and hasattr(self.model, "hid_next") and self.model.hid_next is not None:
#                     with torch.no_grad():
#                         hnext = self.model.hid_next.detach()
#                         if hasattr(self.model, "hid"):
#                             cur_b = hnext.size(1)
#                             self.model.hid[:, :cur_b, :] = hnext.to(self.model.hid.device)
#                         else:
#                             self.model.hid = hnext.detach().clone()
#                 return float(loss)
#             else:
#                 logits, probs = self._model_forward(x, deterministic=True, clip=clip_flag)
#                 b, s, C = logits.shape
#                 logits_flat = logits.contiguous().view(-1, C)
#                 targets_flat = y.contiguous().view(-1)
#                 loss = self.val_loss_fn(logits_flat, targets_flat).item()
#                 if self.hid_prop and hasattr(self.model, "hid_next") and self.model.hid_next is not None:
#                     with torch.no_grad():
#                         hnext = self.model.hid_next.detach()
#                         if hasattr(self.model, "hid"):
#                             cur_b = hnext.size(1)
#                             self.model.hid[:, :cur_b, :] = hnext.to(self.model.hid.device)
#                         else:
#                             self.model.hid = hnext.detach().clone()
#                 return float(loss)

#     # ---------- evaluation helpers over dataset-like objects ----------
#     def evaluate_dataset(self, data, mode="usual", use_all=True):
#         """
#         data: object with methods:
#             - new_epoch() or to_first_batch() (we'll attempt to call both),
#             - get_next_batch() -> (x,y) or (x,y,mask), numpy arrays (same shape as original Theano)
#             - num_batches, data.num_examples or data.data available
#         """
#         # reset pointer
#         if hasattr(data, "to_first_batch"):
#             data.to_first_batch()
#         elif hasattr(data, "new_epoch"):
#             data.new_epoch()

#         if self.hid_prop:
#             # reset hid buffer if model supports it
#             if hasattr(self.model, "hid"):
#                 with torch.no_grad():
#                     self.model.hid.zero_()

#         num_batches = data.num_batches if use_all else min(1, data.num_batches)
#         err = 0.0
#         for i in range(num_batches):
#             batch = data.get_next_batch()
#             # batch is usually (x, y) or (x, y, mask)
#             loss = self.eval_batch(batch[0], batch[1], mode=mode)
#             batch_size_factor = batch[0].shape[1] if self.hid_prop else batch[0].shape[0]
#             err += loss * batch_size_factor

#         if use_all:
#             denom = (data.data.shape[0] - 1) if self.hid_prop else data.num_examples
#             err = err / float(denom)
#         else:
#             last_batch = batch
#             batch_size_factor = last_batch[0].shape[1] if self.hid_prop else last_batch[0].shape[0]
#             err = err / float(num_batches * batch_size_factor)

#         if self.hid_prop:
#             err = float(np.exp(err))
#         # reset pointer
#         if hasattr(data, "to_first_batch"):
#             data.to_first_batch()
#         return float(err)

#     # ---------- training loop ----------
#     def fit(self,
#             train_data,
#             test_data,
#             num_epochs: int = 1,
#             valid_data=None,
#             print_fq: int = 1,
#             save_fq: int = 0,
#             file_name: str = "model.pt",
#             sparsification_eval_fun: Optional[Callable] = None):
#         """
#         High-level training loop similar to Theano train_char_lm_model.
#         train_data/test_data — PTB_word-like objects (they return numpy batches via get_next_batch()).
#         """
        
        
#         # compute train_size if not set
#         if self.train_size is None:
#             self.train_size = train_data.train_size()

#         print("Training ...")
#         for epoch in range(1, int(num_epochs) + 1):
#             start_time = time.time()
#             if hasattr(train_data, "new_epoch"):
#                 train_data.new_epoch()
#             # reset hidden if needed
#             if self.hid_prop and hasattr(self.model, "hid"):
#                 with torch.no_grad():
#                     self.model.hid.zero_()

#             grad_norm_max = 0.0
#             mean_grad_norm = 0.0
#             tr_l = 0.0
            
#             num_batches = len(train_data)
            
#             for batch_idx, (x, y) in enumerate(train_data):
#                 loss_val, grad_norm = self.train_step(x, y)
#                 print(f"batch_index: {batch_idx}/{num_batches}; loss: {loss_val:.4f}")
#                 grad_norm_max = max(grad_norm_max, grad_norm)
#                 mean_grad_norm += grad_norm
#                 tr_l += loss_val * (x.shape[1] if self.hid_prop else x.shape[0])

#             mean_grad_norm = mean_grad_norm / float(num_batches)
#             tr_l = tr_l / float((len(train_data.data) - 1) if self.hid_prop else train_data.num_examples)
#             train_time = time.time()

#             if (epoch) % print_fq == 0 or epoch == num_epochs:
#                 print("Epoch {} took {:.3f}s \t loss = {}, \t grad norm = {},\t{}".
#                       format(epoch, train_time - start_time, tr_l, grad_norm_max, mean_grad_norm))
#                 # print evaluations (Train/Test/Val) in Theano style
#                 print("Train " + self._eval_summary(train_data, use_all=False))
#                 if valid_data:
#                     print("Val " + self._eval_summary(valid_data, use_all=True))
#                 print("Test " + self._eval_summary(test_data, use_all=True))
#                 if sparsification_eval_fun:
#                     sparsification_eval_fun()

#             if save_fq and (epoch) % save_fq == 0:
#                 torch.save(self.model.state_dict(), f"{file_name}_epoch_{epoch}.pt")

#         return self.model

#     def _eval_summary(self, dataset, use_all=True):
#         # returns string similar to print_evaluate
#         parts = []
#         for t in self.test_types:
#             try:
#                 val = self.evaluate_dataset(dataset, mode=t, use_all=use_all)
#             except Exception as e:
#                 val = float('nan')
#             parts.append("%.4f" % val)
#         return "(" + ",".join(self.test_types) + "): " + ", ".join(parts)


In [22]:
def load_PTB_word(ptb_path: str, fname_train: str, fname_val: str, fname_test: str
                 ) -> Tuple[torch.LongTensor, torch.LongTensor, torch.LongTensor]:
    """
    Читает файлы PTB (word-level split by '_' as in original) и возвращает
    train, val, test как 1D torch.LongTensor индексов слов.
    Поведение чтения сохранено: data = fin.read()[::2] (как в оригинале).
    """
    def read_words(fname: str):
        with open(ptb_path + fname, "r") as fin:
            # Поведение оригинала: берем каждый второй символ
            raw = fin.read()[::2]
        words = []
        for token in raw.split():
            # оригинал: разбивает по '_' и добавляет <eos>
            parts = token.split('_')
            words.extend(parts)
            words.append('<eos>')
        return words

    train_words = read_words(fname_train)
    val_words = read_words(fname_val)
    test_words = read_words(fname_test)

    # Собираем словарь по всем трех частям
    vocab = list(set(train_words + val_words + test_words))
    VOCAB_SIZE = len(vocab)
    print("VOCAB_SIZE =", VOCAB_SIZE)

    word_to_id = {w: i for i, w in enumerate(vocab)}
    # id_to_word = {i: w for i, w in enumerate(vocab)}  # при необходимости

    # Переводим последовательности в индексы
    train_idx = [word_to_id[w] for w in train_words]
    val_idx = [word_to_id[w] for w in val_words]
    test_idx = [word_to_id[w] for w in test_words]

    # Возвращаем torch.LongTensor (1D)
    return (torch.tensor(train_idx, dtype=torch.long),
            torch.tensor(val_idx, dtype=torch.long),
            torch.tensor(test_idx, dtype=torch.long))


class PTB_word:
    """
    Аналог оригинального класса, но на torch.Tensor.
    Принимает 1D torch.LongTensor (последовательность индексов).
    Поведение:
      - усечение до кратного batch_size,
      - reshape -> (batch_size, -1), затем transpose -> (num_steps, batch_size)
      - get_next_batch возвращает (batch, seq_len) входы и цели.
    """
    def __init__(self, data: torch.LongTensor, length: int, batch_size: int):
        """
        data: 1D torch.LongTensor
        length: длина последовательности (seq_len)
        batch_size: batch size
        """
        if not torch.is_tensor(data):
            raise TypeError("data must be a torch.Tensor (1D) of long/int")

        if data.dim() != 1:
            raise ValueError("data must be 1D tensor")

        self.length = int(length)
        self.batch_size = int(batch_size)

        # исходная длина (для подсчёта num_batches как в оригинале)
        self.raw_len = data.size(0)

        # усечение до целого числа батчей (умножение на batch_size)
        n_full = (self.raw_len // self.batch_size) * self.batch_size
        truncated = data[:n_full].clone()  # copy to avoid aliasing

        # reshape: (batch_size, -1) then transpose -> (num_cols, batch_size)
        # number of columns = n_full / batch_size
        cols = n_full // self.batch_size
        reshaped = truncated.view(self.batch_size, cols).t().contiguous()  # shape (cols, batch_size)

        self.data = reshaped  # shape (num_rows, batch_size)
        self.batch_ind = 0

        # число батчей на эпоху: ceil(raw_len / (batch_size * length))
        self.num_batches = int(np.ceil(self.raw_len / float(self.batch_size * self.length)))

    def new_epoch(self):
        self.batch_ind = 0

    def to_first_batch(self):
        self.batch_ind = 0

    def get_next_batch(self):
        """
        Возвращает (inputs, targets), оба shape (batch_size, seq_len), dtype long.
        Поведение: берёт seq_len = min(self.length, remaining_rows - 1),
        затем slice rows [batch_ind : batch_ind + seq_len + 1] и возвращает
        batch[:-1].T (входы) и batch[1:].T (цели) — как в оригинале.
        """
        n_rows = self.data.size(0)  # количество временных шагов в колонках
        # если осталось меньше 2 строк, то seq_len может быть 0 -> вернём пустые батчи
        seq_len = min(self.length, max(0, n_rows - self.batch_ind - 1))
        if seq_len <= 0:
            # в оригинале такое, вероятно, не встречалось, но защитимся
            return (torch.empty((self.batch_size, 0), dtype=torch.long),
                    torch.empty((self.batch_size, 0), dtype=torch.long))

        # берём seq_len+1 рядов для сдвига
        chunk = self.data[self.batch_ind:self.batch_ind + seq_len + 1]  # shape (seq_len+1, batch_size)
        self.batch_ind += seq_len

        # входы = все кроме последней строки, цели = все кроме первой
        inp = chunk[:-1].t().contiguous()   # (batch_size, seq_len)
        targ = chunk[1:].t().contiguous()   # (batch_size, seq_len)
        return inp, targ


class PTBWordLoader:
    def __init__(self, ptb_word, vocab_size, device="cpu"):
        self.ptb = ptb_word
        self.vocab_size = vocab_size
        self.device = device
        self.idx_dtype = torch.long
        self.data = ptb_word.data
        self.num_examples = len(self.data)

    def __iter__(self):
        self.ptb.new_epoch()
        for _ in range(self.ptb.num_batches):
            x, y = self.ptb.get_next_batch()   # (batch, seq_len)
            
            def to_tensor(arr, dtype=torch.float32):
                if isinstance(arr, torch.Tensor):
                    t = arr
                    if dtype is not None and t.dtype != dtype:
                        t = t.to(dtype=dtype)
                    return t
                elif isinstance(arr, np.ndarray):
                    t = torch.from_numpy(arr)
                    if dtype is not None:
                        t = t.to(dtype=dtype)
                    return t
            
            x_t = to_tensor(x, dtype=self.idx_dtype)
            y_t = to_tensor(y, dtype=self.idx_dtype)
            x_onehot = nn.functional.one_hot(x_t, num_classes=self.vocab_size).float()  # (batch, seq_len, vocab_size)
            yield x_onehot, y_t

    def __len__(self):
        return int(self.ptb.num_batches)

    def train_size(self):
        return self.ptb.data.numel()


In [23]:
ptb_path = "Data/ptb/"
fnames = ["ptb.char.train.txt", "ptb.char.valid.txt", "ptb.char.test.txt"]

# config = sys.argv[1] if len(sys.argv) > 1 else "LSTM"
config = 'LCCL'
file_name = f"Results/weights_{config}"

n_hidden = 256
seq_len = 35
grad_clip = 10
batch_size = 32
vocab_size = 10000
learning_rate = 0.002
hid_prop = True
num_epoches = 10
save_fq = 50
print_fq = 1
seed = 0

torch.manual_seed(seed)
np.random.seed(seed)

In [24]:
train_data, valid_data, test_data = load_PTB_word(ptb_path, *fnames)
train_data = PTB_word(train_data, seq_len, batch_size)
valid_data = PTB_word(valid_data, seq_len, batch_size)
test_data = PTB_word(test_data, seq_len, batch_size)

train_loader = PTBWordLoader(train_data, vocab_size)
valid_loader = PTBWordLoader(valid_data, vocab_size)
test_loader = PTBWordLoader(test_data, vocab_size)

VOCAB_SIZE = 10000


In [25]:
class CustomCriterion(nn.Module):
    def __init__(self, net, base_criterion):
        super().__init__()
        self.net = net
        self.base_criterion = base_criterion
        
    def forward(self, pred, target):
        train_size = pred.numel()

        base_criterion_loss = self.base_criterion(pred, target)
        
        reg_loss = 0.0
        for module in self.net.modules():
            if hasattr(module, "eval_reg"):
                current_reg_loss = module.eval_reg(train_size)
                reg_loss = reg_loss + current_reg_loss
        
        loss = base_criterion_loss + reg_loss
        return loss
    

In [29]:
lm_net = LMNet(vocab_size, n_hidden, config, hid_prop, batch_size)

criterion = nn.CrossEntropyLoss()
# criterion = CustomCriterion(net=lm_net, base_criterion=nn.CrossEntropyLoss())
optimizer = optim.Adam(lm_net.parameters(), lr=learning_rate)


lm_net = train_char_lm_model(
    lm_net, 
    train_loader, 
    test_loader, 
    optimizer, 
    criterion,
    num_epochs=num_epoches, 
    valid_loader=valid_loader,
    print_fq=print_fq, 
    save_fq=save_fq, 
    file_name=file_name,
    sparsification_eval_fun=lm_net.evaluate_compression if hasattr(lm_net, "evaluate_compression") else None
)


Training ...
epoch: 1/10; batch: 0/830; loss: 0.5189; tokens: 1120
epoch: 1/10; batch: 1/830; loss: 0.4160; tokens: 1120
epoch: 1/10; batch: 2/830; loss: 0.3233; tokens: 1120
epoch: 1/10; batch: 3/830; loss: 0.2418; tokens: 1120
epoch: 1/10; batch: 4/830; loss: 0.1598; tokens: 1120
epoch: 1/10; batch: 5/830; loss: 0.1227; tokens: 1120
epoch: 1/10; batch: 6/830; loss: 0.0463; tokens: 1120
epoch: 1/10; batch: 7/830; loss: 0.0051; tokens: 1120
epoch: 1/10; batch: 8/830; loss: -0.0633; tokens: 1120
epoch: 1/10; batch: 9/830; loss: -0.1397; tokens: 1120
epoch: 1/10; batch: 10/830; loss: -0.2263; tokens: 1120
epoch: 1/10; batch: 11/830; loss: -0.4183; tokens: 1120
epoch: 1/10; batch: 12/830; loss: -0.6000; tokens: 1120
epoch: 1/10; batch: 13/830; loss: -0.8407; tokens: 1120
epoch: 1/10; batch: 14/830; loss: -1.0761; tokens: 1120
epoch: 1/10; batch: 15/830; loss: -1.1576; tokens: 1120
epoch: 1/10; batch: 16/830; loss: -1.3031; tokens: 1120
epoch: 1/10; batch: 17/830; loss: -1.5245; tokens: 11

In [ ]:
# lm_net_2 = LMNet(vocab_size, n_hidden, config, hid_prop, batch_size)


# criterion = nn.CrossEntropyLoss()
# # criterion = CustomCriterion(net=lm_net, base_criterion=nn.CrossEntropyLoss())
# optimizer = optim.Adam(lm_net.parameters(), lr=learning_rate)

# trainer = LMTrainer(lm_net, 
#                     optimizer,
#                     grad_clip=10.0, 
#                     mc_samples=10, 
#                     input_is_indices=False, 
#                     hid_prop=lm_net.hid_prop)

# trainer.fit(train_loader, 
#             test_loader, 
#             num_epochs=num_epoches, 
#             valid_data=valid_loader, 
#             print_fq=1, save_fq=0)


profiling

In [ ]:
import torch.profiler as profiler


def test_with_loader(model, loader):
    model.train()


    # -------------------
    # профилируем одну эпоху
    # -------------------
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CPU],
        record_shapes=True,
        with_stack=True
    ) as prof:
        for step, (x, y) in enumerate(loader):
            with profiler.record_function("model_inference"):
                logits, probs = model(x)

            if step >= 2:  # ограничимся 3 батчами для теста
                break

    print("Output shapes:")
    print("probs:", probs.shape)
    print("logits:", logits.shape)

    # вывод статистики
    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))


model = LMNet(vocab_size, n_hidden, config, hid_prop, batch_size)

test_with_loader(model, train_loader)


Output shapes:
probs: torch.Size([64, 35, 10000])
logits: torch.Size([64, 35, 10000])
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
              model_inference        10.04%     125.364ms        89.32%        1.116s     371.857ms             3  
                 aten::matmul         0.16%       1.955ms        39.42%     492.354ms       1.132ms           435  
                    aten::mul        28.71%     358.583ms        29.02%     362.488ms     208.686us          1737  
                     aten::mm        25.19%     314.598ms        25.20%     314.779ms     723.630us           435  
                  aten::copy_        18.70%     233.559ms        18.70%     233.559ms     269.388us   

In [ ]:

# -----------------------------
# Сохранение
# -----------------------------
torch.save(lm_net.state_dict(), file_name + ".pt")